<a href="https://colab.research.google.com/github/alicsrsustain-sudo/HVAC-Optimization-/blob/main/pump_affinity_law2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import sys
import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────
# INPUTS
# ─────────────────────────────────────────────
CSV_PATH            = "History 2025-12-18T03_15_01 to 2026-06-22T07_45_02.csv"   # Path to the PEAK CSV export
CURRENT_SPEED_PCT   = 95           # % of full speed
NEW_SPEED_PCT       = 85           # % of full speed (proposed)
PUMP_RATED_POWER_KW = 30          # Pump motor rated power in kW (nameplate)
MOTOR_EFFICIENCY    = 0.91        # From nameplate
ELECTRICITY_COST_PER_KWH = 0.2225  # £/kWh — UK average (from PEAK)

# ─────────────────────────────────────────────
def load_operating_hours(csv_path: str, on_speed_pct: int = 95) -> tuple[float, float, str, str]:
    """
    Read a PEAK CSV export and return:
      (hours_in_dataset, annualised_hours_per_year, date_start, date_end)

    The CSV records pump speed at regular intervals (15-min here).
    Rows where Value == on_speed_pct are counted as ON; Value == 0 is OFF.
    Annualised hours scale the dataset total up to a full year.
    """
    path = Path(csv_path)
    if not path.exists():
        sys.exit(f"ERROR: CSV file not found — {csv_path}")

    df = pd.read_csv(csv_path)

    # Identify the timestamp and value columns (handle PEAK's quoted header)
    ts_col    = next(c for c in df.columns if "Timestamp" in c)
    value_col = "Value"

    df[ts_col] = pd.to_datetime(df[ts_col])

    # Detect the sample interval in minutes from consecutive timestamps
    intervals = df[ts_col].diff().dropna().dt.total_seconds() / 60
    sample_interval_min = intervals.mode()[0]          # most common gap

    # Count ON rows only (speed == CURRENT_SPEED_PCT, i.e. not 0%)
    on_rows = df[df[value_col] == on_speed_pct]
    hours_in_dataset = len(on_rows) * (sample_interval_min / 60)

    # Date range of the dataset
    date_start = df[ts_col].min().strftime("%d %b %Y")
    date_end   = df[ts_col].max().strftime("%d %b %Y")

    # Annualise: scale to a full year
    total_days = (df[ts_col].max() - df[ts_col].min()).days
    if total_days == 0:
        sys.exit("ERROR: Dataset contains only a single timestamp.")

    annualised_hours = hours_in_dataset * (365 / total_days)

    return hours_in_dataset, annualised_hours, date_start, date_end, sample_interval_min, total_days


# ─────────────────────────────────────────────
# STEP 2 — Physics: affinity / cube law
# ─────────────────────────────────────────────
def electrical_power_kw(rated_kw: float, speed_pct: float, efficiency: float) -> float:
    """
    Electrical input power at a given speed using the Affinity (Cube) Law.
    shaft_power  = rated_power × (speed / 100)³
    electrical   = shaft_power / motor_efficiency
    """
    shaft_kw     = rated_kw * (speed_pct / 100) ** 3
    electrical_kw = shaft_kw / efficiency
    return electrical_kw


# ─────────────────────────────────────────────
# STEP 3 — Run
# ─────────────────────────────────────────────
(hours_dataset,
 hours_annual,
 date_start,
 date_end,
 interval_min,
 days_dataset) = load_operating_hours(CSV_PATH, on_speed_pct=CURRENT_SPEED_PCT)

power_current_kw = electrical_power_kw(PUMP_RATED_POWER_KW, CURRENT_SPEED_PCT, MOTOR_EFFICIENCY)
power_new_kw     = electrical_power_kw(PUMP_RATED_POWER_KW, NEW_SPEED_PCT,     MOTOR_EFFICIENCY)

power_saved_kw      = power_current_kw - power_new_kw
power_reduction_pct = (power_saved_kw / power_current_kw) * 100

energy_current_kwh  = power_current_kw * hours_annual
energy_new_kwh      = power_new_kw     * hours_annual
energy_saved_kwh    = energy_current_kwh - energy_new_kwh

cost_current        = energy_current_kwh * ELECTRICITY_COST_PER_KWH
cost_new            = energy_new_kwh     * ELECTRICITY_COST_PER_KWH
cost_saved          = cost_current - cost_new

co2_saved_kg        = energy_saved_kwh * CO2_FACTOR_KG_PER_KWH
co2_saved_tonnes    = co2_saved_kg / 1_000

# ─────────────────────────────────────────────
# STEP 4 — Report
# ─────────────────────────────────────────────
SEP  = "=" * 65
SEP2 = "-" * 65

print(SEP)
print("   HVAC PUMP SPEED REDUCTION — SAVINGS REPORT")
print(SEP)

print(f"   Date range                 : {date_start} → {date_end}  ({days_dataset} days)")
print(f"   Sample interval            : {interval_min:.0f} min")
print(f"   Pump ON hours in dataset   : {hours_dataset:,.1f} h  (speed = {CURRENT_SPEED_PCT}%)")

print(f"\n  SYSTEM CONFIGURATION")
print(f"   Pump rated power           : {PUMP_RATED_POWER_KW:.1f} kW")
print(f"   Motor efficiency           : {MOTOR_EFFICIENCY*100:.0f}%")
print(f"   Electricity tariff         : £{ELECTRICITY_COST_PER_KWH:.4f}/kWh")

print(f"\n  POWER CONSUMPTION")
print(f"   Current speed              : {CURRENT_SPEED_PCT}%  →  {power_current_kw:.2f} kW")
print(f"   Proposed speed             : {NEW_SPEED_PCT}%  →  {power_new_kw:.2f} kW")
print(f"   Power saved                : {power_saved_kw:.2f} kW  ({power_reduction_pct:.1f}% reduction)")

print(f"\n  ANNUAL ENERGY SAVINGS  (annualised from {days_dataset}-day dataset)")
print(f"   Energy at current speed    : {energy_current_kwh:,.0f} kWh/year")
print(f"   Energy at proposed speed   : {energy_new_kwh:,.0f} kWh/year")
print(f"    Energy saved             : {energy_saved_kwh:,.0f} kWh/year")

print(f"\n  ANNUAL COST SAVINGS")
print(f"   Cost at current speed      : £{cost_current:,.2f}/year")
print(f"   Cost at proposed speed     : £{cost_new:,.2f}/year")
print(f"    Money saved              : £{cost_saved:,.2f}/year")

print(f"\n{SEP}")

   HVAC PUMP SPEED REDUCTION — SAVINGS REPORT
   Date range                 : 18 Dec 2025 → 22 Jun 2026  (186 days)
   Sample interval            : 15 min
   Pump ON hours in dataset   : 338.8 h  (speed = 95%)

  SYSTEM CONFIGURATION
   Pump rated power           : 30.0 kW
   Motor efficiency           : 91%
   Electricity tariff         : £0.2225/kWh

  POWER CONSUMPTION
   Current speed              : 95%  →  28.27 kW
   Proposed speed             : 85%  →  20.25 kW
   Power saved                : 8.02 kW  (28.4% reduction)

  ANNUAL ENERGY SAVINGS  (annualised from 186-day dataset)
   Energy at current speed    : 18,789 kWh/year
   Energy at proposed speed   : 13,458 kWh/year
    Energy saved             : 5,331 kWh/year

  ANNUAL COST SAVINGS
   Cost at current speed      : £4,180.61/year
   Cost at proposed speed     : £2,994.51/year
    Money saved              : £1,186.10/year

